# 1. Calorie Expenditure Model Training

Train and package a regression model for Kaggle Playground Series S5E5: Predict Calorie Expenditure.

This notebook is designed to produce everything needed for the backend calorie expenditure agent:

- A compact model artifact suitable for repo promotion.
- A metrics JSON file that can be reviewed before promotion.
- A feature schema JSON file for backend inference alignment.
- A Kaggle `submission.csv` when `test.csv` is available.
- A final zip archive containing all generated artifacts.

Artifacts are written to `/kaggle/working/calorie_expenditure_artifacts`.


In [1]:
from __future__ import annotations

import json
import math
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TARGET_COLUMN = "Calories"
ARTIFACT_DIR = Path("/kaggle/working/calorie_expenditure_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Locate Kaggle Files

This step searches common Kaggle input paths so the notebook works whether the competition dataset is attached directly or copied into another input folder.

Expected files:

- `train.csv` is required.
- `test.csv` is optional but required for creating `submission.csv`.


In [2]:
def find_dataset_file(filename: str) -> Path:
    candidates = list(Path("/kaggle/input").glob(f"**/{filename}"))
    if candidates:
        return candidates[0]
    local_candidate = Path(filename)
    if local_candidate.exists():
        return local_candidate
    raise FileNotFoundError(f"Could not find {filename}. Attach the Kaggle competition data first.")


train_path = find_dataset_file("train.csv")
test_path = find_dataset_file("test.csv") if list(Path("/kaggle/input").glob("**/test.csv")) else None

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path) if test_path else None

print(f"Train path: {train_path}")
print(f"Train shape: {train_df.shape}")
if test_df is not None:
    print(f"Test path: {test_path}")
    print(f"Test shape: {test_df.shape}")

train_df.head()


Train path: /kaggle/input/competitions/playground-series-s5e5/train.csv
Train shape: (750000, 9)
Test path: /kaggle/input/competitions/playground-series-s5e5/test.csv
Test shape: (250000, 8)


,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,0,male,36,189.0,82.0,26.0,101.0,41.0,150.0
1,1,female,64,163.0,60.0,8.0,85.0,39.7,34.0
2,2,female,51,161.0,64.0,7.0,84.0,39.8,29.0
3,3,male,20,192.0,90.0,25.0,105.0,40.7,140.0
4,4,female,38,166.0,61.0,25.0,102.0,40.6,146.0


## 3. Prepare Features

The competition target is expected to be `Calories`. The model ignores `id` as a feature and preserves it only for submission output.

The exported feature schema should be shared with the backend so online inference uses the same column names and preprocessing assumptions as training.


In [ ]:
if TARGET_COLUMN not in train_df.columns:
    raise ValueError(f"Expected target column {TARGET_COLUMN!r}; found {train_df.columns.tolist()}")

id_columns = [column for column in ["id", "Id", "ID"] if column in train_df.columns]
feature_columns = [column for column in train_df.columns if column not in id_columns + [TARGET_COLUMN]]

X = train_df[feature_columns].copy()
y = train_df[TARGET_COLUMN].astype(float).clip(lower=0)

categorical_columns = []
for column in feature_columns:
    dtype = X[column].dtype
    if (
        pd.api.types.is_object_dtype(dtype)
        or pd.api.types.is_bool_dtype(dtype)
        or pd.api.types.is_string_dtype(dtype)
        or isinstance(dtype, pd.CategoricalDtype)
    ):
        categorical_columns.append(column)
numeric_columns = [column for column in feature_columns if column not in categorical_columns]

try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", one_hot_encoder),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_columns),
        ("cat", categorical_pipeline, categorical_columns),
    ],
    remainder="drop",
)

feature_schema = {
    "target_column": TARGET_COLUMN,
    "id_columns": id_columns,
    "feature_columns": feature_columns,
    "numeric_columns": numeric_columns,
    "categorical_columns": categorical_columns,
}

feature_schema


## 4. Train Compact Baseline Models

This section trains compact baseline models and compares them on a validation split. The target is transformed with `log1p` and restored with `expm1`, which usually behaves better for positive calorie targets and aligns with RMSLE-style evaluation.

The notebook intentionally avoids large random forest artifacts. A full 300-tree forest on this dataset can exceed several GB after serialization, which is too large for practical repo promotion.

Selection rule: choose the model with the lowest validation `rmsle` among compact candidates.


In [ ]:
def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.maximum(np.asarray(y_true, dtype=float), 0)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 0)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))


def build_model(regressor):
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", regressor),
        ]
    )
    return TransformedTargetRegressor(
        regressor=pipeline,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )


X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

candidate_models = {
    "hist_gradient_boosting_fast": build_model(
        HistGradientBoostingRegressor(
            learning_rate=0.08,
            max_iter=180,
            max_leaf_nodes=31,
            l2_regularization=0.02,
            random_state=RANDOM_STATE,
        )
    ),
    "hist_gradient_boosting_deep": build_model(
        HistGradientBoostingRegressor(
            learning_rate=0.05,
            max_iter=260,
            max_leaf_nodes=63,
            l2_regularization=0.01,
            random_state=RANDOM_STATE,
        )
    ),
}

results = {}
trained_models = {}

for model_name, model in candidate_models.items():
    model.fit(X_train, y_train)
    valid_pred = np.maximum(model.predict(X_valid), 0)
    results[model_name] = {
        "mae": float(mean_absolute_error(y_valid, valid_pred)),
        "rmse": float(math.sqrt(mean_squared_error(y_valid, valid_pred))),
        "rmsle": rmsle(y_valid, valid_pred),
        "r2": float(r2_score(y_valid, valid_pred)),
    }
    trained_models[model_name] = model

metrics_df = pd.DataFrame(results).T.sort_values("rmsle")
metrics_df


## 5. Refit Best Compact Model and Save Artifacts

The best validation model is refit on the full training dataset, then saved with compression, metrics, and feature schema.

Share these files back to the repo when you are ready to promote the model:

- `calorie_expenditure_model.joblib`
- `metrics.json`
- `feature_schema.json`


In [ ]:
best_model_name = metrics_df.index[0]
best_model = candidate_models[best_model_name]
best_model.fit(X, y)

model_path = ARTIFACT_DIR / "calorie_expenditure_model.joblib"
metrics_path = ARTIFACT_DIR / "metrics.json"
schema_path = ARTIFACT_DIR / "feature_schema.json"

# Compression keeps the artifact practical to download and commit if needed.
joblib.dump(best_model, model_path, compress=("xz", 3))

model_size_mb = model_path.stat().st_size / (1024 * 1024)
metrics_payload = {
    "competition": "playground-series-s5e5",
    "target": TARGET_COLUMN,
    "best_model_name": best_model_name,
    "validation_metrics": results,
    "selected_metric": "rmsle",
    "random_state": RANDOM_STATE,
    "train_rows": int(len(train_df)),
    "feature_count": int(len(feature_columns)),
    "artifact_size_mb": round(model_size_mb, 2),
    "artifact_policy": "Compact boosted models only; avoid full random forest artifacts.",
}

metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
schema_path.write_text(json.dumps(feature_schema, indent=2), encoding="utf-8")

print(f"Best model: {best_model_name}")
print(f"Saved model: {model_path}")
print(f"Model size: {model_size_mb:.2f} MB")
print(f"Saved metrics: {metrics_path}")
print(f"Saved schema: {schema_path}")
if model_size_mb > 100:
    print("Warning: model artifact is larger than 100 MB. Consider reducing max_iter or max_leaf_nodes before promoting to the repo.")
metrics_payload


## 6. Create Submission

When `test.csv` is available, this step creates a Kaggle-ready `submission.csv` using the selected model.


In [6]:
if test_df is not None:
    missing_features = [column for column in feature_columns if column not in test_df.columns]
    if missing_features:
        raise ValueError(f"Test set is missing features: {missing_features}")

    test_predictions = np.maximum(best_model.predict(test_df[feature_columns]), 0)
    submission_id_column = id_columns[0] if id_columns and id_columns[0] in test_df.columns else None
    submission_df = pd.DataFrame(
        {
            "id": test_df[submission_id_column] if submission_id_column else np.arange(len(test_df)),
            TARGET_COLUMN: test_predictions,
        }
    )
    submission_path = ARTIFACT_DIR / "submission.csv"
    submission_df.to_csv(submission_path, index=False)
    print(f"Saved submission: {submission_path}")
    display(submission_df.head())
else:
    print("No test.csv found, so submission.csv was not created.")


Saved submission: /kaggle/working/calorie_expenditure_artifacts/submission.csv


,id,Calories
0,750000,27.106733
1,750001,106.520559
2,750002,87.546094
3,750003,126.577916
4,750004,75.380879


## 7. Quick Inference Check

This final model smoke test reloads the saved artifact and predicts on a few training rows. It is a lightweight check that serialization and inference both work before packaging.


In [7]:
loaded_model = joblib.load(model_path)
sample_predictions = np.maximum(loaded_model.predict(X.head(5)), 0)
pd.DataFrame({"actual": y.head(5).to_numpy(), "predicted": sample_predictions})


,actual,predicted
0,150.0,149.509227
1,34.0,34.579925
2,29.0,29.012058
3,140.0,138.811874
4,146.0,145.511985


## 8. Package Artifacts for Download

This step zips every file in the artifact directory into one archive. Download this zip from Kaggle and place the model, metrics, and schema into the repo when promoting the trained model.


In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

zip_path = Path("/kaggle/working/calorie_expenditure_artifacts.zip")

with ZipFile(zip_path, mode="w", compression=ZIP_DEFLATED) as archive:
    for artifact_path in sorted(ARTIFACT_DIR.glob("*")):
        if artifact_path.is_file():
            archive.write(artifact_path, arcname=artifact_path.name)

packaged_files = sorted(path.name for path in ARTIFACT_DIR.glob("*") if path.is_file())
zip_size_mb = zip_path.stat().st_size / (1024 * 1024)
print(f"Created artifact bundle: {zip_path}")
print(f"Zip size: {zip_size_mb:.2f} MB")
print("Packaged files:")
for file_name in packaged_files:
    print(f"- {file_name}")
if zip_size_mb > 100:
    print("Warning: artifact zip is larger than 100 MB. Do not commit it directly; use a release or artifact store.")

zip_path
